# Module 7 — Batch query the deployed compound agent

We've built the agent; now we'll exercise it the way a downstream
analyst pipeline would: a Delta table of questions in, a Delta table of
answers out. We use `ai_query` so this scales horizontally — no Python
loop, no rate-limiting headaches, just SQL that fans questions across
the serving endpoint.

Questions cover both cyber threat-intel **and** network operations
topics — the agent picks the right tools (Genie / Knowledge Assistant /
CVE lookup / netops outages) per question.

In [ ]:
%run ./_config


In [ ]:
# Wait for the agent endpoint to be Ready before sending traffic
from databricks.sdk import WorkspaceClient
import time

w = WorkspaceClient()
for attempt in range(60):
    ep = w.serving_endpoints.get(AGENT_ENDPOINT)
    state = (ep.state.ready.value if ep.state and ep.state.ready else 'UNKNOWN')
    cfg = (ep.state.config_update.value if ep.state and ep.state.config_update else 'NONE')
    print(f"  [{attempt:2}] ready={state} config_update={cfg}")
    if state == 'READY':
        break
    time.sleep(15)
else:
    raise RuntimeError(f'Endpoint {AGENT_ENDPOINT} did not become READY in 15 minutes')
print(f'\nEndpoint {AGENT_ENDPOINT} is ready.')


## Build the question set

Mix of cyber threat-intel and network ops. The agent will route each to
the right tool: structured questions go to Genie, document/policy questions
go to the Knowledge Assistant, outage questions go to `netops_outages_query`.

In [ ]:
questions = [
    # --- Cyber threat intel ---
    ('cyber',  'What are the top 5 vendors in the CISA KEV catalog by number of exploited CVEs?'),
    ('cyber',  'List the 3 most recent KEV entries and explain the recommended action for each.'),
    ('cyber',  'Which DoD assets are running products that have any CVE in the KEV catalog?'),
    ('cyber',  'Summarize the latest CISA advisory in the corpus and the affected products.'),
    ('cyber',  'Are there any critical (CVSS >= 9.0) network-related CVEs published recently?'),
    ('cyber',  'What MITRE ATT&CK techniques are commonly associated with initial-access exploitation of edge VPN appliances?'),
    # --- Network ops ---
    ('netops', 'What carriers have had the most outage reports in the last 30 days?'),
    ('netops', 'Have there been any recent BGP-related outages and what is the typical root cause?'),
    ('netops', 'Show me critical-severity outages from the last 7 days and their summaries.'),
    ('netops', 'Are there any DNS-related outages mentioned and which providers were affected?'),
    # --- Cross-domain (the interesting ones) ---
    ('cross',  'Are any of the recent network outages potentially correlated with active CVEs in the KEV catalog?'),
    ('cross',  'Have any of the providers reporting outages been affected by recent CISA advisories?'),
]

from pyspark.sql import Row
df_q = spark.createDataFrame([Row(category=c, question=q) for c,q in questions])
df_q.write.mode('overwrite').saveAsTable(f'{BASE}.agent_questions')
df_q.display()


## Fan questions across the agent endpoint with `ai_query`

`ai_query` calls the serving endpoint per row in parallel. The agent is a
ResponsesAgent, so we pass a chat-style request and unwrap the response.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {BASE}.agent_answers AS
WITH raw AS (
  SELECT
    category,
    question,
    ai_query(
      '{AGENT_ENDPOINT}',
      named_struct(
        'input', array(named_struct('role', 'user', 'content', question))
      ),
      failOnError => false
    ) AS response
  FROM {BASE}.agent_questions
)
SELECT
  category,
  question,
  response.errorMessage AS error,
  response.result AS raw_result,
  COALESCE(
    parse_json(response.result):output[size(parse_json(response.result):output)-1].content[0].text::string,
    parse_json(response.result):output[0].content[0].text::string,
    parse_json(response.result):choices[0].message.content::string,
    response.result
  ) AS answer_text
FROM raw
""")

spark.table(f'{BASE}.agent_answers').select('category','question','error','answer_text').display()


## Sanity: count answers per category and flag failures

In [ ]:
spark.sql(f"""
SELECT
  category,
  COUNT(*) AS asked,
  SUM(CASE WHEN answer_text IS NULL OR length(answer_text) < 20 THEN 1 ELSE 0 END) AS failed_or_empty
FROM {BASE}.agent_answers
GROUP BY category
ORDER BY category
""").display()
